# Correcting the GPS coordinate errors in matched_stops.csv

The matched_stops.csv file has errors GPS coordinates, where latitude and longitude have high values ​​that are outside of Sweden.

By analyzing the data, we found that the same bus stop (identified by stopLabel) sometimes appears with correct coordinates and sometimes with incorrect ones. The incorrect coordinates are the correct ones multiplied by either 12 or 18.

<p style="color: green;">Our approach:</p>

1. Split the data into rows with valid coordinates and rows with invalid coordinates
2. Build a lookup table from the valid rows, mapping each stop name to its correct coordinates
3. For each invalid row, look up the stop name in the lookup table and replace the coordinates
4. For stops not found in the lookup table, detect the correct scaling factor (12 or 18) by checking which one produces coordinates within Sweden's bounds
5. Save the corrected data to data/processed/matched_stops_corrected.csv

## Setup: Imports And Loading Data

In [2]:
import pandas as pd
from pathlib import Path


dataRaw = Path("../data/raw")

matched_stops = pd.read_csv(dataRaw/'matched_stops.csv', encoding_errors='ignore')

## Correction Of matched_stops.csv

We split the data into two parts, rows with correct coordinates and rows with incorrect ones.

In [3]:
# correct gps coordinates (2676)
valid = matched_stops[(matched_stops['latitude'] <= 57.36406) & (matched_stops['longitude'] <= 16.145123)]

# wrong gps coordinates (16478)
invalid = matched_stops[~((matched_stops['latitude'] <= 57.36406) & (matched_stops['longitude'] <= 16.145123))]

# check
print(len(valid))
print(len(invalid))

2676
16478


Build a lookup table from the valid rows

In [4]:
validTable = valid[['stopLabel', 'latitude', 'longitude']]

validTable = validTable.drop_duplicates('stopLabel')

lookup = validTable.set_index('stopLabel').to_dict('index')

#lookup

len(lookup)

64

Checking if there are missed stopLabels in the invalid data

In [5]:
missing = invalid[~invalid['stopLabel'].isin(lookup.keys())]
len(missing)

786

786 rows must be corrected via the scaling factor (divide by 12 or 18)

Now we are going to correct the invalid data by going through each line in invalid, if the stop name is in the lookup table we replace lat/lon with the values ​​from the table. If it doesn't exist we divide lat/lon by 12, check if the result is within Halland (sweden) borders (lat: 55–58, lon: 12–14), otherwise divide by 18.

In [ ]:
def correct(row):
    stopName = row['stopLabel']

    if stopName in lookup:
        row['latitude'] = lookup[stopName]['latitude']
        row['longitude'] = lookup[stopName]['longitude']
    else:
        correctedLat = row['latitude']/12
        correctedLon = row['longitude']/12

        if ((55 <= correctedLat <= 58) and (12 <= correctedLon <= 14)):
            row['latitude'] = correctedLat
            row['longitude'] = correctedLon

        else:
            correctedLat2 = row['latitude']/14
            correctedLon2 = row['longitude']/14
            
            if ((55 <= correctedLat2 <= 58) and (12 <= correctedLon2 <= 14)):
                row['latitude'] = correctedLat2
                row['longitude'] = correctedLon2
        
            else:
                row['latitude'] = row['latitude']/18
                row['longitude'] = row['longitude']/18

    return row

corrected_invalid = invalid.apply(correct, axis=1)

#corrected_invalid[['stopLabel', 'latitude', 'longitude']].to_csv('test.csv')

Found that some of the rows where multiplied with 14

Now we put together valid and corrected_invalid

In [26]:
corrected = pd.concat([valid, corrected_invalid]).sort_index().to_csv('../data/processed/matched_stops_corrected.csv')

The GPS data seems to be correct now.

<p style="color: red;">
    total row? swedish letters?
</p>